# OpenDistillation v0 Demo Skeleton

> A personal model factory for the AI PC and AI phone era.

This notebook is the first runnable skeleton for one narrow model type: a notes / school model from TXT/MD notes. It covers text upload/loading, validation, chunking, dataset schema validation, deterministic mock teacher generation, an optional short student fine-tuning entry point, and an optional before/after comparison.

By default it does not train a model, download a model, use a GPU, call paid APIs, or export GGUF files. The training cell stays skipped until you explicitly opt in from a Colab GPU runtime.

## Runtime setup

Run this notebook from the repository root locally. In Colab, opening the notebook from GitHub starts in `/content`, so the setup cell clones the OpenDistillation repository before importing local helpers. The default path uses only Python standard-library code and local helper modules. Optional training requires extra Hugging Face packages and a GPU.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

OPEN_DISTILLATION_REPO_URL = "https://github.com/tacotuesday8888/OpenDistillation.git"


def is_colab_runtime():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "opendistillation").exists() and (PROJECT_ROOT.parent / "src" / "opendistillation").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src" / "opendistillation").exists():
    if is_colab_runtime():
        PROJECT_ROOT = Path("/content/OpenDistillation")
        if not (PROJECT_ROOT / "src" / "opendistillation").exists():
            subprocess.check_call(["git", "clone", "--depth", "1", OPEN_DISTILLATION_REPO_URL, str(PROJECT_ROOT)])
    else:
        raise RuntimeError("OpenDistillation source files were not found. Run this notebook from the repository root, or open the GitHub notebook in Colab.")

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from opendistillation import (
    BeforeAfterComparisonEngine,
    MockTeacherEngine,
    OPTIONAL_TRAINING_PACKAGES,
    TeacherRequest,
    SFTLoRAConfig,
    SFTLoRATrainingEngine,
    build_comparison_request,
    build_pip_install_command,
    build_training_request,
    check_training_runtime,
    chunk_text,
    explain_runtime_failure,
    format_runtime_check,
    load_text_document,
    rows_to_jsonl,
)

print(f"Using project root: {PROJECT_ROOT}")
print("Runtime: mock/data-prep path runs on CPU. Optional training is skipped unless RUN_TRAINING is set to True later.")

## Optional dependency install

Keep `INSTALL_TRAINING_DEPS = False` for the default local demo path. In Colab, switch to a GPU runtime first, then set this to `True` once to install the optional Hugging Face training packages.

In [ ]:
INSTALL_TRAINING_DEPS = False

print("Optional training packages:")
print(", ".join(OPTIONAL_TRAINING_PACKAGES))
print("Install command:")
print(build_pip_install_command())

if INSTALL_TRAINING_DEPS:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *OPTIONAL_TRAINING_PACKAGES])
    print("Optional training dependencies installed. Restart the runtime if Colab asks, then rerun setup.")
else:
    print("Install skipped. Keep this false for the local CPU demo path.")

## Upload or load a TXT/MD file

In Colab, this cell offers a file upload. Outside Colab, it uses `examples/sample-notes.md` so the notebook can run top to bottom without any external service.

In [ ]:
def load_uploaded_or_sample():
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        sample_path = PROJECT_ROOT / "examples" / "sample-notes.md"
        return sample_path.name, sample_path.read_text(encoding="utf-8")

    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded. Upload one .txt or .md file to continue.")
    filename, content = next(iter(uploaded.items()))
    return filename, content


filename, content = load_uploaded_or_sample()
document = load_text_document(filename, content)

print(f"File: {document.filename}")
print(f"Extension: {document.extension}")
print(f"Characters: {document.char_count}")
print(f"Approx. words: {document.word_count}")
if document.warnings:
    print("Warnings:")
    for warning in document.warnings:
        print(f"- {warning}")
print("\nPreview:\n")
print(document.preview)

## Chunk the document

The v0 chunker prefers paragraph boundaries, preserves source order, removes empty chunks, and assigns stable IDs like `chunk-0001`.

In [ ]:
chunks = chunk_text(document.text, max_chars=300)
print(f"Chunks: {len(chunks)}")

for chunk in chunks[:3]:
    print("=" * 72)
    print(f"{chunk.id} | chars={chunk.char_count} | words={chunk.word_count}")
    print(chunk.text[:500])

## Generate mock training examples

This skeleton uses `MockTeacherEngine`, a deterministic local teacher path. It does not send text to a remote endpoint. Later goals can replace this engine with real open-source teacher backends while keeping the same notebook flow.

In [ ]:
teacher_engine = MockTeacherEngine()
request = TeacherRequest(chunks=chunks, examples_per_chunk=2)
rows = teacher_engine.generate(request)
dataset_jsonl = rows_to_jsonl(rows)

print(f"Teacher engine: {teacher_engine.name}")
print(f"Sends text to remote endpoint: {teacher_engine.sends_data_remote}")
print(f"Generated examples: {len(rows)}")
print("\nFirst 5 examples:\n")
for row in rows[:5]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

## Dataset JSONL preview

The initial schema is one JSON object per line with exactly these fields: `instruction`, `response`, and `source_chunk_id`.

In [ ]:
print("First JSONL lines:\n")
print("\n".join(dataset_jsonl.splitlines()[:5]))

# Save to the runtime temp directory, not the repository, so generated data is not committed.
output_path = Path(tempfile.gettempdir()) / "opendistillation_mock_training_data.jsonl"
output_path.write_text(dataset_jsonl, encoding="utf-8")
print(f"Saved runtime dataset to {output_path}")

try:
    from google.colab import files  # type: ignore
except ImportError:
    print("Download helper is available only in Colab; use the path above locally.")
else:
    files.download(str(output_path))

## Optional short student fine-tuning

This is the first real training entry point, but it is off by default. It uses one small student model, `Qwen/Qwen2.5-0.5B-Instruct`, with TRL `SFTTrainer` and PEFT LoRA.

Leave `RUN_TRAINING = False` when you want the notebook to run without GPU or model downloads. Set it to `True` only in a Colab GPU runtime after installing `torch`, `transformers`, `datasets`, `trl`, `peft`, and `accelerate`. Output goes under `outputs/`, which is ignored by git.

In [ ]:
RUN_TRAINING = False

training_config = SFTLoRAConfig()
training_request = build_training_request(
    rows,
    output_dir=PROJECT_ROOT / "outputs" / "notes-lora",
    config=training_config,
)
training_engine = SFTLoRATrainingEngine(training_config)
training_plan = training_engine.describe(training_request)

print("Training plan:")
for key, value in training_plan.items():
    print(f"- {key}: {value}")

training_result = None
if RUN_TRAINING:
    runtime_check = check_training_runtime()
    print("Runtime check:")
    for line in format_runtime_check(runtime_check):
        print(f"- {line}")

    if not runtime_check.can_run_training:
        raise RuntimeError("Training runtime is not ready. Install missing packages or switch to a GPU runtime, then rerun this cell.")

    try:
        training_result = training_engine.train(training_request)
    except Exception as exc:
        print("Training failed with a recoverable setup/runtime issue.")
        for line in explain_runtime_failure(exc):
            print(f"- {line}")
        raise

    print(f"Training engine: {training_result.engine_name}")
    print(f"Adapter output: {training_result.output_path}")
    for note in training_result.notes:
        print(f"- {note}")
else:
    print("Training skipped. Set RUN_TRAINING = True in a Colab GPU runtime to start the short SFT run.")

## Before/after comparison

This comparison uses the first generated dataset question. It runs only after the optional training cell creates a LoRA adapter. The result is a qualitative sanity check, not a benchmark.

In [ ]:
if training_result is None:
    print("Before/after comparison skipped because training did not run.")
else:
    comparison_request = build_comparison_request(rows, training_result, config=training_config)
    comparison_engine = BeforeAfterComparisonEngine()
    comparison_plan = comparison_engine.describe(comparison_request)

    print("Comparison plan:")
    for key, value in comparison_plan.items():
        print(f"- {key}: {value}")

    try:
        comparison = comparison_engine.compare(comparison_request)
    except Exception as exc:
        print("Before/after comparison failed with a recoverable setup/runtime issue.")
        for line in explain_runtime_failure(exc):
            print(f"- {line}")
        raise

    print("\nQuestion:")
    print(comparison.question)
    print("\nReference answer from generated dataset:")
    print(comparison.reference_response)
    print("\nBase model answer:")
    print(comparison.base_answer)
    print("\nTrained adapter answer:")
    print(comparison.trained_answer)
    for note in comparison.notes:
        print(f"- {note}")

## Manual Colab smoke-test checklist

Use this checklist before marking the GPU path verified: GPU runtime selected, optional dependency install succeeds, runtime check prints a GPU name, model download starts, short training starts, adapter output prints under `outputs/notes-lora/adapter`, before/after comparison prints both answers, total runtime is recorded, and any memory failure is recorded with the exact error.

## Export placeholder

GGUF export and local runtime instructions are later milestones. The intended `ExportEngine` plug-in point is after training output exists: convert or merge the model output, then document llama.cpp and/or Ollama-style local commands.

In [ ]:
print("Export placeholder: skipped.")
print("No GGUF files, model artifacts, or local runtime files are created by this skeleton.")